# Seminar Notebook 3.2: Automated Dictionary Methods

**LSE MY459: Computational Text Analysis and Large Language Models** (WT 2026)

**Ryan Hübert**

This notebook covers automated dictionary methods (ADMs) for measuring concepts like sentiment in text, including VADER, custom dictionaries, and the Lexicoder Sentiment Dictionary (LSD).

## Set up steps


### Directory management

We begin with some directory management to specify the file path to the folder on your computer where you wish to store data for this notebook.

In [ ]:
import os
sdir = os.path.join(os.path.expanduser("~"), "LSE-MY459-WT26", "SeminarWeek07")
corpus_path = os.path.join(sdir, "candidate-corpus.csv")
if not os.path.exists(corpus_path):
    raise FileNotFoundError("You must run the first notebook before proceeding!")

### Loading the corpus

We need to load the candidate tweets corpus we created in notebook `00-make-dfm.ipynb`. Dictionary methods typically work on the original text rather than the preprocessed DFM, so we load the corpus CSV file.

In [ ]:
import pandas as pd

corpus = pd.read_csv(corpus_path, dtype="object")
print(corpus.shape)
corpus.head()

Let's create an object `ocols` that is a list of the original columns in the dataset.

In [ ]:
ocols = list(corpus.columns)
ocols

## VADER Sentiment Analysis

VADER (Valence Aware Dictionary and sEntiment Reasoner) is a sentiment analysis tool designed for social media text. Unlike simple dictionary methods that assign +1/-1 to words, VADER uses weighted sentiment scores, accounts for capitalisation and punctuation, and handles negations and intensifiers automatically. It is available in `nltk`.

VADER returns four scores: `neg`, `neu`, and `pos` (proportions), and `compound` (overall score from -1 to +1). Let's test it on some sample sentences.

In [ ]:
test_sentences = [
    "I absolutely LOVE MY459!",
    "I absolutely love MY459.",
    "This is okay.",
    "This is terrible and I hate it."
]

Notice that VADER gives higher scores for capitalised "LOVE" than lowercase "love". The `compound` score is typically interpreted as: positive if >= 0.05, negative if <= -0.05, and neutral otherwise.

### Applying VADER to the corpus

We can visualise the distribution of VADER scores.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.hist(corpus["vader_compound"], bins=50, edgecolor="black", alpha=0.7)
plt.axvline(0.05, color="green", linestyle="--", label="Positive threshold")
plt.axvline(-0.05, color="red", linestyle="--", label="Negative threshold")
plt.xlabel("VADER Compound Score")
plt.ylabel("Frequency")
plt.title("Distribution of VADER Compound Scores")
plt.legend()
plt.show()

## Creating a Custom Dictionary

We begin by creating a simple dictionary to detect policy-related language in these political tweets.

In [ ]:
policy_dict = {
    "economy": ["econom*", "job*", "unemploy*", "tax*", "budget", "deficit",
                "inflation", "wage*", "income", "trade", "tariff*"],
    "healthcare": ["health*", "medical", "hospital*", "doctor*", "patient*",
                   "insurance", "medicare", "medicaid", "prescription*", "drug*"],
    "immigration": ["immigra*", "border*", "migrant*", "refugee*", "asylum",
                    "visa*", "deportat*", "citizenship"],
    "environment": ["climat*", "environment*", "pollution", "emission*", "carbon",
                    "renewable", "solar", "wind", "green", "clean energy"]
}

This simple dictionary uses glob-style wildcards (`*`), which adds flexibility when applying the dictionary. For example, the words unemployment and unemployed will both be considered "economy" words since `unemploy*` matches both. However, in Python we'll use regular expressions via the `re` module to do pattern matching. So, we need to convert the glob wildcard `*` to its regular expression equivalent `\w*`.

We then combine all patterns for each dictionary key into a _single_ compiled regular expression. This allows us to do very efficient matching.

Let's test on a sample sentence.

In [ ]:
test_policy = "We need to create jobs and fix the broken healthcare system. Immigration reform is essential."

Now we apply the custom dictionary to the full corpus. To do this, we will add new columns to the `corpus` object which give the number of matches for each category in the dictionary.

We can now measure how much of each document corresponds to the different categories, as well as whether a document had any mention of a topic.

## The Lexicoder Sentiment Dictionary (LSD)

The Lexicoder Sentiment Dictionary was designed for sentiment analysis in political texts and has been validated with human-coded news content. It is available at <https://www.snsoroka.com/data-lexicoder>. Unlike VADER (which handles everything internally), LSD is a raw dictionary that we need to apply ourselves. This gives us more control and transparency, but requires more work. The dictionary contains four word lists: `positive`, `negative`, `neg_positive` (negated positive phrases like "not good", which count as negative), and `neg_negative` (negated negative phrases like "not bad", which count as positive).

We first download and extract the LSD dictionary files.

The LSD comes in `.lc3` format. Let's find and examine the files.

Now, we parse the LSD.

The file uses `+` to indicate category headers, followed by word patterns. The `*` character is a wildcard that matches any suffix (e.g., `abandon*` matches "abandon", "abandoned", "abandoning", etc.). We now parse the files to extract the word lists.

Let's see what one of these lists looks like:

### Converting patterns to regular expressions

The LSD uses glob-style wildcards (`*`), but Python's `re` module uses regex syntax. We need to convert `*` to `\w*` and escape special characters. We then combine all patterns into a single regex for efficient matching.

Let's see what one of these regex patterns looks like:

### Applying LSD to a single document

Before applying LSD to the full corpus, let's walk through the process step-by-step on a single document. We'll use a simple test sentence first.

We preprocess by lowercasing and removing punctuation except apostrophes and hyphens.

We calculate the document length for later normalisation.

We will extract all the words/phrases in the text matching values in the dictionary and store them in an object `s` (for "sentiment"):

We must check for negated phrases first, before checking for regular positive/negative words. This avoids double counting (e.g., "not great" should count as negated positive, not as positive).

We remove the matched negated phrases from the text before searching for regular positive/negative words.

Now we search for regular positive and negative words in the remaining text.

We calculate the sentiment score. Positive signals include positive words and negated negative phrases. Negative signals include negative words and negated positive phrases.

Let's also try this on a real document from our corpus.

### Applying LSD to the full corpus

Now that we understand the process, we can wrap it in a function and apply it to all documents.

Let's examine the distribution of LSD scores.

## For fun: compare LSD and VADER

Let's see how the two methods correlate with each other.

### Saving the results

We save the corpus with all sentiment and policy scores for use in later analysis.